# 4. 기초통계와 회귀

웹 강의자료: [https://statjeong.github.io/moj-python-lecture/04_statistics.html](https://statjeong.github.io/moj-python-lecture/04_statistics.html)

이 장에서는 기술통계, 교차표, 두 집단 평균 비교, 선형회귀, 로지스틱 회귀를 실행해 봅니다. 실습 데이터가 10명뿐이라 결과의 통계적 의미보다는 코드가 어떻게 돌아가고 결과표를 어디서 읽는지에 집중합니다.

## 4.1 준비

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
import statsmodels.formula.api as smf

주소 = "https://raw.githubusercontent.com/StatJeong/moj-python-lecture/main/data/"
수용자 = pd.read_csv(주소 + "inmates.csv")
수용자["징벌여부"] = (수용자["징벌횟수"] > 0).astype(int)

## 4.2 기술통계

In [ ]:
수용자[["나이", "형기_개월", "상담횟수", "징벌횟수"]].describe().round(2)

| 행 | 뜻 |
|---|---|
| count | 값이 있는 칸 수 (상담횟수는 결측 1개로 9) |
| mean, std | 평균, 표준편차 |
| min, max | 최솟값, 최댓값 |
| 25%, 50%, 75% | 사분위수 (50%가 중앙값) |

평균과 중앙값을 따로 구할 수도 있습니다.

In [ ]:
print(수용자["나이"].mean())
print(수용자["나이"].median())

## 4.3 교차표

범주형 변수 두 개의 관계는 교차표로 봅니다.

In [ ]:
pd.crosstab(수용자["직업훈련"], 수용자["징벌여부"], margins=True)

행마다의 비율로 보려면 `normalize="index"`를 씁니다.

In [ ]:
pd.crosstab(수용자["직업훈련"], 수용자["징벌여부"], normalize="index").round(2)

## 4.4 두 집단 평균 비교 (t-검정)

직업훈련 참여자와 비참여자의 평균 징벌횟수를 비교합니다.

In [ ]:
참여 = 수용자[수용자["직업훈련"] == "Y"]["징벌횟수"]
비참여 = 수용자[수용자["직업훈련"] == "N"]["징벌횟수"]

print("참여 평균:", 참여.mean())
print("비참여 평균:", 비참여.mean())

결과 = stats.ttest_ind(참여, 비참여, equal_var=False)
print(결과)

`pvalue`가 0.05보다 작으면 두 집단 평균이 다르다고 판단하는 것이 일반적입니다. 다만 집단마다 5명이라 이 결과는 연습용으로만 봅니다.

## 4.5 선형회귀

상담횟수가 많을수록 징벌횟수가 어떻게 달라지는지 봅니다.

### 모형식

수용자 $i$의 징벌횟수를 상담횟수의 일차식과, 그 식으로 설명되지 않는 나머지로 나누어 씁니다.

$$
\text{징벌횟수}_i = \beta_0 + \beta_1 \times \text{상담횟수}_i + \varepsilon_i, \qquad i = 1, 2, \dots, n
$$

| 기호 | 뜻 |
|---|---|
| $\beta_0$ | 절편. 상담횟수가 0일 때의 평균 징벌횟수 |
| $\beta_1$ | 기울기. 상담횟수가 1회 늘 때 평균 징벌횟수가 변하는 양 |
| $\varepsilon_i$ | 오차. 상담횟수로 설명되지 않는 나머지 (평균 0) |
| $n$ | 분석에 쓰인 사람 수 |

$\beta_0$, $\beta_1$은 모르는 값이고, 데이터로 추정한 값은 $b_0$, $b_1$로 씁니다.

파이썬에서는 이 모형식을 `"징벌횟수 ~ 상담횟수"`로 줄여 씁니다. `~` 왼쪽이 결과, 오른쪽이 설명변수이고, 절편($\beta_0$)과 오차($\varepsilon_i$)는 따로 적지 않아도 자동으로 들어갑니다.

### 실행과 결과표

In [ ]:
모형1 = smf.ols("징벌횟수 ~ 상담횟수", data=수용자).fit()
print(모형1.summary())

결과표에서 볼 곳은 다음과 같습니다.

| 위치 | 기호 | 뜻 |
|---|---|---|
| `No. Observations` | $n$ | 분석에 쓰인 관측치 수 |
| `Intercept` 줄의 `coef` | $b_0$ | 절편의 추정값 (2.7757) |
| `상담횟수` 줄의 `coef` | $b_1$ | 기울기의 추정값 (-0.4206) |
| `std err` | | 추정값의 표준오차 |
| P>\|t\| | | 계수가 0인지 검정한 p값 |
| `R-squared` | $R^2$ | 모형이 설명하는 변동의 비율 |

관측치 수가 10이 아니라 9입니다. 상담횟수가 비어 있는 1007번이 자동으로 빠졌기 때문입니다. 계수만 따로 꺼낼 수도 있습니다.

In [ ]:
print(모형1.nobs)
print(모형1.params)

결과표의 `coef`를 모형식의 $\beta_0$, $\beta_1$ 자리에 넣으면 추정된 식이 됩니다.

$$
\widehat{\text{징벌횟수}} = 2.776 - 0.421 \times \text{상담횟수}
$$

모자 기호(^)는 실제 값이 아니라 식으로 계산한 예측값이라는 표시입니다. 오차 $\varepsilon_i$는 평균이 0이므로 예측식에는 쓰지 않습니다.

상담횟수의 계수가 음수이므로, 이 데이터에서는 상담횟수가 1회 많을수록 징벌횟수가 평균 약 0.42회 적습니다. 이것은 함께 움직이는 관계(연관)이고, 상담이 징벌을 줄였다는 인과관계를 뜻하지는 않습니다.

> [팁] 식 안에 숫자로 시작하거나 띄어쓰기가 있는 열 이름
>
> `smf.ols("수용률 ~ 1일평균수용인원", ...)`처럼 숫자로 시작하는 열 이름을 식에 그대로 쓰면 `SyntaxError: invalid syntax`가 납니다. 열 이름을 `Q("...")`로 감쌉니다.
>
> ```python
> smf.ols('수용률 ~ Q("1일평균수용인원")', data=표).fit()
> ```
>
> 띄어쓰기가 있는 열 이름도 같은 방법으로 씁니다.


### 예측값과 잔차

추정된 식에 상담횟수를 넣으면 예측값이 나옵니다. 상담을 5회 받은 사람의 예측 징벌횟수는 다음과 같습니다(반올림 차이를 줄이려고 소수 다섯째 자리까지 씁니다).

$$
\begin{aligned}
\widehat{\text{징벌횟수}} &= 2.77570 - 0.42056 \times 5 \\
&= 2.77570 - 2.10280 \\
&= 0.67290
\end{aligned}
$$

In [ ]:
b0 = 모형1.params["Intercept"]
b1 = 모형1.params["상담횟수"]
print(b0 + b1 * 5)

1002번은 상담 5회, 징벌 0회입니다. 실제 값에서 예측값을 뺀 것을 잔차라고 합니다.

$$
\text{잔차} = \text{실제 값} - \text{예측값} = 0 - 0.6729 = -0.6729
$$

### 계수를 정하는 방법: 최소제곱법

상담횟수를 $x_i$, 징벌횟수를 $y_i$라고 쓰겠습니다. $b_0$, $b_1$은 잔차를 제곱해 더한 값(SSE)이 가장 작아지도록 정합니다.

$$
\text{SSE}(b_0, b_1) = \sum_{i=1}^{n} \left( y_i - b_0 - b_1 x_i \right)^2
$$

이 값을 가장 작게 만드는 $b_0$, $b_1$은 다음과 같습니다($\bar{x}$, $\bar{y}$는 평균).

$$
b_1 = \frac{\sum_{i=1}^{n} (x_i - \bar{x})(y_i - \bar{y})}{\sum_{i=1}^{n} (x_i - \bar{x})^2},
\qquad
b_0 = \bar{y} - b_1 \bar{x}
$$

> [참고] 공식을 유도하는 과정 (펼쳐 보기)
>
> **1단계: $b_0$에 대해 미분해 0으로 놓기**
>
> SSE가 가장 작은 곳에서는 $b_0$에 대한 기울기(미분)가 0입니다. 제곱을 미분하면 $2 \times (\text{안쪽}) \times (\text{안쪽을 } b_0\text{로 미분한 값} = -1)$입니다.
>
> $$
> \begin{aligned}
> \frac{\partial\, \text{SSE}}{\partial b_0} = \sum_{i=1}^{n} 2\left( y_i - b_0 - b_1 x_i \right)(-1) &= 0 \\
> \sum_{i=1}^{n} \left( y_i - b_0 - b_1 x_i \right) &= 0 && (\text{양변을 } {-2}\text{로 나눔}) \\
> \sum y_i - n b_0 - b_1 \sum x_i &= 0 && (\textstyle\sum b_0 = n b_0) \\
> \bar{y} - b_0 - b_1 \bar{x} &= 0 && (\text{양변을 } n\text{으로 나눔}) \\
> b_0 &= \bar{y} - b_1 \bar{x}
> \end{aligned}
> $$
>
> **2단계: $b_1$에 대해 미분해 0으로 놓기**
>
> 이번에는 안쪽을 $b_1$로 미분한 값이 $-x_i$입니다.
>
> $$
> \begin{aligned}
> \frac{\partial\, \text{SSE}}{\partial b_1} = \sum_{i=1}^{n} 2\left( y_i - b_0 - b_1 x_i \right)(-x_i) &= 0 \\
> \sum_{i=1}^{n} x_i \left( y_i - b_0 - b_1 x_i \right) &= 0
> \end{aligned}
> $$
>
> 1단계의 $b_0 = \bar{y} - b_1 \bar{x}$를 넣으면 괄호 안이 $y_i - \bar{y} + b_1 \bar{x} - b_1 x_i = (y_i - \bar{y}) - b_1 (x_i - \bar{x})$가 됩니다.
>
> $$
> \begin{aligned}
> \sum x_i \left\{ (y_i - \bar{y}) - b_1 (x_i - \bar{x}) \right\} &= 0 \\
> \sum x_i (y_i - \bar{y}) &= b_1 \sum x_i (x_i - \bar{x})
> \end{aligned}
> $$
>
> **3단계: $x_i$를 $(x_i - \bar{x})$로 바꾸기**
>
> 편차의 합은 0입니다: $\sum (y_i - \bar{y}) = \sum y_i - n\bar{y} = 0$. 그래서 $\bar{x} \sum (y_i - \bar{y}) = 0$을 빼도 값이 같습니다.
>
> $$
> \sum x_i (y_i - \bar{y}) = \sum x_i (y_i - \bar{y}) - \bar{x} \sum (y_i - \bar{y}) = \sum (x_i - \bar{x})(y_i - \bar{y})
> $$
>
> 같은 방법으로 $\sum x_i (x_i - \bar{x}) = \sum (x_i - \bar{x})^2$입니다. 2단계 식의 양변을 바꿔 쓰고 $\sum (x_i - \bar{x})^2$로 나누면 다음과 같습니다.
>
> $$
> b_1 = \frac{\sum (x_i - \bar{x})(y_i - \bar{y})}{\sum (x_i - \bar{x})^2}
> $$
>
> **4단계: 실습 데이터의 숫자 넣기**
>
> 상담횟수가 있는 9명의 값은 다음과 같습니다.
>
> | | 1001 | 1002 | 1003 | 1004 | 1005 | 1006 | 1008 | 1009 | 1010 | 합계 |
> |---|---|---|---|---|---|---|---|---|---|---|
> | $x_i$ (상담) | 2 | 5 | 6 | 1 | 8 | 3 | 4 | 2 | 7 | 38 |
> | $y_i$ (징벌) | 2 | 0 | 0 | 3 | 0 | 1 | 1 | 2 | 0 | 9 |
> | $x_i y_i$ | 4 | 0 | 0 | 3 | 0 | 3 | 4 | 4 | 0 | 18 |
> | $x_i^2$ | 4 | 25 | 36 | 1 | 64 | 9 | 16 | 4 | 49 | 208 |
>
> 평균은 $\bar{x} = 38/9$, $\bar{y} = 9/9 = 1$입니다. 괄호를 풀면 $\sum (x_i - \bar{x})(y_i - \bar{y}) = \sum x_i y_i - \bar{y}\sum x_i - \bar{x}\sum y_i + n\bar{x}\bar{y} = \sum x_i y_i - n\bar{x}\bar{y}$이고, 같은 방법으로 $\sum (x_i - \bar{x})^2 = \sum x_i^2 - n\bar{x}^2$입니다.
>
> $$
> \begin{aligned}
> \sum (x_i - \bar{x})(y_i - \bar{y}) &= 18 - 9 \times \frac{38}{9} \times 1 = 18 - 38 = -20 \\
> \sum (x_i - \bar{x})^2 &= 208 - 9 \times \left(\frac{38}{9}\right)^2 = 208 - \frac{1444}{9} = \frac{428}{9} \\
> b_1 &= \frac{-20}{428/9} = -\frac{180}{428} = -\frac{45}{107} \approx -0.42056 \\
> b_0 &= 1 - \left(-\frac{45}{107}\right) \times \frac{38}{9} = 1 + \frac{190}{107} = \frac{297}{107} \approx 2.77570
> \end{aligned}
> $$
>
> 결과표의 `coef`(2.7757, -0.4206)와 같습니다.


공식을 pandas로 그대로 계산해 결과표의 계수와 비교해 봅니다.

In [ ]:
자료 = 수용자.dropna()          # 빈칸이 있는 1007번을 뺀 9명
x = 자료["상담횟수"]
y = 자료["징벌횟수"]

b1 = ((x - x.mean()) * (y - y.mean())).sum() / ((x - x.mean()) ** 2).sum()
b0 = y.mean() - b1 * x.mean()
print("b0:", b0)
print("b1:", b1)

### 결정계수(R²)

$R^2$는 징벌횟수의 전체 변동 가운데 식이 설명하는 비율입니다.

$$
R^2 = 1 - \frac{\sum (y_i - \hat{y}_i)^2}{\sum (y_i - \bar{y})^2} = 1 - \frac{\text{식으로 설명하지 못한 변동}}{\text{전체 변동}}
$$

이 데이터에서는 전체 변동이 $\sum (y_i - \bar{y})^2 = 10$, 설명하지 못한 변동이 약 1.589라서 $R^2 = 1 - 1.589 / 10 = 0.841$입니다. 징벌횟수 변동의 약 84%를 상담횟수의 일차식으로 설명한다는 뜻입니다. 9명뿐인 연습용 데이터라 실제 업무 자료에서는 이렇게 높게 나오기 어렵습니다.

## 4.6 로지스틱 회귀

결과가 0 / 1인 경우(징벌을 받았는지 여부)에는 로지스틱 회귀를 씁니다.

### 모형식

수용자 $i$가 징벌을 받을 확률(징벌여부가 1일 확률)을 $p_i$라고 하면 모형식은 다음과 같습니다.

$$
\log \frac{p_i}{1 - p_i} = \beta_0 + \beta_1 \times \text{나이}_i
$$

| 기호 | 뜻 |
|---|---|
| $p_i$ | 징벌을 받을 확률 (0과 1 사이) |
| $\dfrac{p_i}{1-p_i}$ | 오즈. 받을 확률이 안 받을 확률의 몇 배인지. 예: $p = 0.8$이면 $0.8 / 0.2 = 4$ |
| $\log$ | 자연로그 ($e$를 밑으로 하는 로그) |

확률 $p_i$를 선형회귀처럼 일차식으로 바로 놓으면 계산값이 0보다 작거나 1보다 커질 수 있습니다. 오즈에 로그를 씌운 값(로그 오즈)은 음수부터 양수까지 어떤 값이든 될 수 있어서 일차식과 맞출 수 있습니다.

### 확률로 풀어 쓰기

일차식 부분을 $\eta = \beta_0 + \beta_1 \times \text{나이}$로 줄여 쓰고, 모형식을 $p$에 대해 풉니다.

$$
\begin{aligned}
\log \frac{p}{1 - p} &= \eta \\
\frac{p}{1 - p} &= e^{\eta} && (\text{양변에 지수함수를 씌움}) \\
p &= e^{\eta} (1 - p) = e^{\eta} - p\, e^{\eta} && (\text{양변에 } 1 - p \text{를 곱함}) \\
p + p\, e^{\eta} &= e^{\eta} && (p\, e^{\eta} \text{를 왼쪽으로 옮김}) \\
p \,(1 + e^{\eta}) &= e^{\eta} \\
p &= \frac{e^{\eta}}{1 + e^{\eta}} = \frac{1}{1 + e^{-\eta}} && (\text{분자와 분모를 } e^{\eta} \text{로 나눔})
\end{aligned}
$$

$\eta$가 아무리 크거나 작아도 $p$는 0과 1 사이에 있습니다.

### 실행과 결과표

> [참고] 실행 결과 위에 나오는 Optimization terminated successfully
>
> 로지스틱 회귀의 `.fit()`은 계산 과정을 몇 줄 출력합니다. 오류가 아니라 계산이 끝났다는 안내입니다. 보이지 않게 하려면 `.fit(disp=0)`을 씁니다.

In [ ]:
모형2 = smf.logit("징벌여부 ~ 나이", data=수용자).fit()
print(모형2.summary())

`Intercept` 줄의 `coef`(7.6907)가 $b_0$, `나이` 줄의 `coef`(-0.1909)가 $b_1$입니다. 추정된 식은 다음과 같습니다.

$$
\log \frac{\hat{p}}{1 - \hat{p}} = 7.691 - 0.191 \times \text{나이}
\qquad \Longleftrightarrow \qquad
\hat{p} = \frac{1}{1 + e^{-(7.691 - 0.191 \times \text{나이})}}
$$

30세와 50세의 예측 확률을 계산해 봅니다.

$$
\begin{aligned}
\text{30세:} \quad \eta &= 7.69072 - 0.19091 \times 30 = 7.69072 - 5.72730 = 1.96342, \\
\hat{p} &= \frac{1}{1 + e^{-1.96342}} = \frac{1}{1 + 0.14038} \approx 0.877 \\
\text{50세:} \quad \eta &= 7.69072 - 0.19091 \times 50 = 7.69072 - 9.54549 = -1.85477, \\
\hat{p} &= \frac{1}{1 + e^{1.85477}} = \frac{1}{1 + 6.39030} \approx 0.135
\end{aligned}
$$

In [ ]:
b0 = 모형2.params["Intercept"]
b1 = 모형2.params["나이"]
for 나이 in [30, 50]:
    일차식 = b0 + b1 * 나이
    확률 = 1 / (1 + np.exp(-일차식))
    print(나이, "세의 예측 확률:", round(확률, 3))

### 오즈비

나이가 1세 많은 사람과 비교해 오즈가 몇 배인지 계산합니다. 모형식에서 오즈는 $\dfrac{p}{1-p} = e^{b_0 + b_1 \times \text{나이}}$입니다.

$$
\begin{aligned}
\frac{\text{오즈}(\text{나이} + 1)}{\text{오즈}(\text{나이})}
&= \frac{e^{\,b_0 + b_1 (\text{나이} + 1)}}{e^{\,b_0 + b_1 \times \text{나이}}} \\
&= e^{\,\{b_0 + b_1 (\text{나이} + 1)\} - \{b_0 + b_1 \times \text{나이}\}} && (e^{a} / e^{c} = e^{a - c}) \\
&= e^{\,b_1} \\
&= e^{-0.1909} \approx 0.826
\end{aligned}
$$

나이가 식에서 지워지므로, 몇 살이든 1세 많아질 때마다 오즈가 0.826배가 됩니다. $1 - 0.826 = 0.174$이므로 징벌을 받을 오즈가 약 17% 낮아진다고 읽습니다. 파이썬에서는 계수 전체에 `np.exp`를 씌워 한 번에 계산합니다.

In [ ]:
np.exp(모형2.params)

`나이` 줄이 오즈비 0.826입니다. `Intercept` 줄(약 2188)은 나이가 0세일 때의 오즈라 해석하지 않습니다.

오즈비는 나이와 관계없이 같지만, 확률의 변화는 나이마다 다릅니다. 위에서 30세의 0.877이 50세에 0.135로 줄었는데, 이 20년 동안 확률이 해마다 같은 크기로 줄어드는 것은 아닙니다.

## 4.7 결과를 믿으면 안 되는 경우

이번에는 직업훈련 참여 여부로 징벌여부를 설명해 봅니다.

In [ ]:
모형3 = smf.logit("징벌여부 ~ 직업훈련", data=수용자).fit()
print(모형3.params)
print(모형3.bse)

코드는 끝까지 실행되지만 경고 메시지가 나오고, 계수가 20을 넘고 표준오차(`bse`)가 1만이 넘는 값으로 나옵니다. 4.3절의 교차표를 다시 보면 직업훈련 N인 5명이 모두 징벌을 받았습니다. 이렇게 한 집단의 결과가 전부 같으면 계수를 추정할 수 없습니다(완전 분리).

모형식으로 보면 이유가 드러납니다. 직업훈련은 Y / N 글자라서 statsmodels가 0 / 1 변수로 바꿔 넣습니다. 결과의 `직업훈련[T.Y]`는 Y이면 1, N이면 0인 변수 $D_i$이고, N이 기준입니다.

$$
\log \frac{p_i}{1 - p_i} = \beta_0 + \beta_1 \times D_i,
\qquad
D_i = \begin{cases} 1 & \text{직업훈련 Y} \\ 0 & \text{직업훈련 N} \end{cases}
$$

| 집단 | 식에 $D_i$를 넣으면 | 실제 비율 | 로그 오즈 |
|---|---|---|---|
| N ($D_i = 0$) | $\beta_0$ | 5명 중 5명, $p = 1$ | 오즈가 $1/0$이라 무한대 |
| Y ($D_i = 1$) | $\beta_0 + \beta_1$ | 5명 중 1명, $p = 0.2$ | $\log(0.2 / 0.8) \approx -1.386$ |

N 집단에 맞추려면 $\beta_0$이 끝없이 커져야 하므로, 계산은 적당한 곳에서 멈추고 큰 값을 내놓습니다. 그 값은 statsmodels 버전에 따라 달라집니다. 반면 Y 집단의 $\beta_0 + \beta_1$은 제대로 맞춰져 있습니다.

In [ ]:
print(모형3.params.sum())     # b0 + b1
print(np.log(0.2 / 0.8))

오류 없이 실행됐다고 해서 결과가 맞는 것은 아닙니다. 경고 메시지와 비정상적으로 큰 값은 확인하고 넘어가야 합니다. 5장에서 AI가 준 코드를 쓸 때도 같은 원칙이 적용됩니다.

## 4.8 연습문제

1. 시설별 나이의 평균과 표준편차를 구하세요.

In [ ]:
# 여기에 코드를 작성해 보세요

2. 징벌횟수를 나이로 설명하는 선형회귀를 실행하고 나이의 계수를 확인하세요.

In [ ]:
# 여기에 코드를 작성해 보세요